# 15 — Chain Rule and Computation Graphs

**Master syllabus coverage:** V2 2.6, 3.3

## Why this matters

Backpropagation is the chain rule organized for a graph with reused intermediate values. Seeing upstream gradients and accumulation now makes later autograd implementation mechanical.

## Learning objectives

- Perform a manual forward and reverse pass through scalar operations.
- Accumulate gradient contributions at graph branches.
- Explain reverse mode as vector–Jacobian products.
- Verify manual derivatives against PyTorch autograd.

Work through the notebook in order. Predict shapes and results before running each code cell. An assertion failure is part of the lesson: read it before changing code.


## 1. The chain rule composes sensitivities

If $y=f(u)$ and $u=g(x)$, then

$$\frac{dy}{dx}=\frac{dy}{du}\frac{du}{dx}.$$

A computation graph breaks a complex function into primitive local operations. The
forward pass saves intermediate values. The reverse pass multiplies each local
derivative by the derivative arriving from downstream—its **upstream gradient**.


In [ ]:
import math
import torch

# Forward: L = (w*x + b - target)^2
x, target = 2.0, 5.0
w, b = 1.5, -0.5
z = w * x
prediction = z + b
error = prediction - target
loss = error**2

# Reverse: start with dL/dL = 1 and walk backward.
dloss_dloss = 1.0
dloss_derror = dloss_dloss * 2 * error
dloss_dprediction = dloss_derror * 1.0
dloss_dz = dloss_dprediction * 1.0
dloss_db = dloss_dprediction * 1.0
dloss_dw = dloss_dz * x
dloss_dx = dloss_dz * w

print("loss:", loss, "dL/dw:", dloss_dw, "dL/db:", dloss_db, "dL/dx:", dloss_dx)


## 2. Branches add gradient contributions

When one value influences the loss through multiple paths, the multivariable chain
rule **sums** contributions. For $L=x^2+3x$, the two paths give $2x$ and $3$, so
$dL/dx=2x+3$. Autograd engines accumulate into `.grad`; overwriting loses paths.


In [ ]:
x_value = 4.0
path_square = 2 * x_value
path_linear = 3.0
total_gradient = path_square + path_linear
assert total_gradient == 11.0

x_torch = torch.tensor(x_value, requires_grad=True)
branched_loss = x_torch**2 + 3 * x_torch
branched_loss.backward()
assert x_torch.grad.item() == total_gradient
print("branch contributions:", path_square, "+", path_linear, "=", total_gradient)


## 3. Vector–Jacobian products

A vector function $y=f(x)$ has Jacobian $J_{ij}=\partial y_i/\partial x_j$.
Reverse-mode autodiff does not normally materialize the full Jacobian. Given upstream
row vector $v^\top=\partial L/\partial y$, it computes $v^\top J$ efficiently. This is
ideal for ML because the final loss is scalar while parameters are numerous.


In [ ]:
x = torch.tensor([2.0, 3.0], dtype=torch.float64, requires_grad=True)

def vector_fn(value: torch.Tensor) -> torch.Tensor:
    return torch.stack((value[0] * value[1], value[0] ** 2 + value[1]))

jacobian = torch.autograd.functional.jacobian(vector_fn, x)
upstream = torch.tensor([0.5, -2.0], dtype=torch.float64)
expected_vjp = upstream @ jacobian
actual_vjp, = torch.autograd.grad(vector_fn(x), x, grad_outputs=upstream)
torch.testing.assert_close(actual_vjp, expected_vjp)
print("Jacobian:\n", jacobian)
print("VJP:", actual_vjp)


## 4. Verify the manual graph

PyTorch records operations involving tensors that require gradients. Calling
`backward()` performs a reverse topological traversal and accumulates leaf gradients.
Gradients accumulate across calls by design, so training loops must clear them.


In [ ]:
w_t = torch.tensor(w, dtype=torch.float64, requires_grad=True)
b_t = torch.tensor(b, dtype=torch.float64, requires_grad=True)
x_t = torch.tensor(x if isinstance(x, float) else 2.0, dtype=torch.float64, requires_grad=True)
torch_loss = (w_t * x_t + b_t - target) ** 2
torch_loss.backward()

assert math.isclose(w_t.grad.item(), dloss_dw)
assert math.isclose(b_t.grad.item(), dloss_db)
assert math.isclose(x_t.grad.item(), dloss_dx)
print("manual reverse pass matches PyTorch")


## Failure modes to recognize

- **Missing branch contribution:** a reused value receives only one path's gradient.
- **Wrong local derivative:** every upstream result before that node may look correct while earlier gradients fail.
- **Stale accumulated gradients:** updates contain contributions from earlier batches.
- **Detached graph:** conversion or an in-place operation removes the differentiable path.

These are diagnostic patterns, not trivia. When a future model fails, reduce the problem to the smallest example in this notebook and test the relevant invariant.


## Deliberate practice

1. Manually backpropagate through `sigmoid(w*x+b)` with a squared-error loss.
2. Draw the graph for `x*x + x` and explain why the two appearances of x contribute separately.
3. Compute one Jacobian explicitly and verify two different VJPs without constructing it in the VJP calculation.

Do not merely rerun the provided cells. Make a prediction, change one variable, and write down why the result did or did not match your prediction.

## Exit ticket

**You are ready to continue when:** you can walk a graph backward, state each local derivative, and sum gradients wherever paths meet.

**Next:** 16 — Probability and statistics.
